In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from sklearn.metrics import roc_auc_score
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

In [ ]:

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        prompt = str(self.data.loc[index, 'prompt'])
        label = int(self.data.loc[index, 'label'])

        encoding = self.tokenizer.encode_plus(
            prompt,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = model.to(device)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss().to(device)

In [ ]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler):
    model.train()
    total_loss = 0
    correct_predictions = 0
    all_labels = []
    all_preds = []

    for data in data_loader:
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        labels = data['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        logits = outputs.logits
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='binary')
    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader), f1

In [ ]:
def eval_model(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    all_labels = []
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for data in data_loader:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            labels = data['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]  # Assuming binary classification
            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='binary')
    cm = confusion_matrix(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    return correct_predictions.double() / len(data_loader.dataset), total_loss / len(data_loader), f1, cm, auc


In [ ]:
EPOCHS = 6
train_df = pd.read_csv('/content/training_data_binary.csv') #Insert Path to training data
val_df = pd.read_csv('/content/training_data_binary.csv') #Insert Path to val data

In [ ]:
train_df.rename(columns={'jailbreak': 'label'}, inplace=True)
val_df.rename(columns={'jailbreak': 'label'}, inplace=True)
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
train_dataset = TextDataset(train_df, tokenizer, max_len=128)
val_dataset = TextDataset(val_df, tokenizer, max_len=128)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16)
optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
total_steps = len(train_loader) * 5
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
train_f1s = []
val_f1s = []
confusion_matrices = []
train_aucs = []
val_aucs = []
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    train_acc, train_loss, train_f1 = train_epoch(model, train_loader, loss_fn, optimizer, device, scheduler)
    val_acc, val_loss, val_f1, val_cm, val_auc = eval_model(model, val_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)
    train_f1s.append(train_f1)
    val_f1s.append(val_f1)
    confusion_matrices.append(val_cm)
    val_aucs.append(val_auc)

    print(f'Train loss {train_loss}, accuracy {train_acc}, F1 score {train_f1}')
    print(f'Validation loss {val_loss}, accuracy {val_acc}, F1 score {val_f1}')
    print(f'Validation AUC: {val_auc}')
    print(f'Confusion Matrix:\n{val_cm}')


In [ ]:
tn, fp, fn, tp = val_cm.ravel()
fpr = fp / (fp + tn)
tnr = tn / (tn + fp)